# Explore HS codes for UK critical and growth minerals

Scratch notebook: search the HMRC commodity lookup for each mineral and pick candidate codes.
Codes you settle on go into `config/minerals.toml`, with notes on anything ambiguous.

Run `python pipeline/fetch_commodity_lookup.py` first so the lookup file exists in `data/raw/`.

In [1]:
import json
from pathlib import Path

import pandas as pd

# Works whether the notebook runs from notebooks/ or from the repo root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Load the most recent lookup pull (file names are dated, so the last one sorted is the newest)
lookup_file = sorted((REPO_ROOT / "data" / "raw").glob("hmrc_commodity_lookup_*.json"))[-1]
with open(lookup_file, encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

print(lookup_file.name, df.shape)
df.head()

hmrc_commodity_lookup_2026-10-12.json (16243, 10)


,CommodityId,Cn8Code,Hs2Code,Hs4Code,Hs6Code,Hs2Description,Hs4Description,Hs6Description,SitcCommodityCode,Cn8LongDescription
0,0,00,NaN,NaN,NaN,NaN,NaN,NaN,0,-
1,1,01,01,NaN,NaN,Live animals,NaN,NaN,0,-
2,2,02,02,NaN,NaN,Meat and edible meat offal,NaN,NaN,0,-
3,3,03,03,NaN,NaN,"Fish and crustaceans, molluscs and other aquat...",NaN,NaN,0,-
4,4,04,04,NaN,NaN,Dairy produce; birds' eggs; natural honey; edi...,NaN,NaN,0,-


In [2]:
# Lower-level descriptions often omit the mineral name, so search all description levels together
TEXT_COLS = ["Hs4Description", "Hs6Description", "Cn8LongDescription"]
text = df[TEXT_COLS].fillna("").agg(" ".join, axis=1)

SHOW_COLS = ["CommodityId", "Cn8Code", "Hs6Code", "Hs4Description", "Hs6Description", "Cn8LongDescription"]

def search(term):
    """Return every lookup row whose descriptions mention the term (case-insensitive)."""
    return df.loc[text.str.contains(term, case=False), SHOW_COLS].sort_values("Cn8Code")

In [3]:
search("cobalt")

,CommodityId,Cn8Code,Hs6Code,Hs4Description,Hs6Description,Cn8LongDescription
4049,26050000,26050000,260500,Cobalt ores and concentrates,Cobalt ores and concentrates,Cobalt ores and concentrates
4071,26171000,26171000,261710,"Ores and concentrates (excl. iron, manganese, ...",Antimony ores and concentrates,Antimony ores and concentrates
4072,26179000,26179000,261790,"Ores and concentrates (excl. iron, manganese, ...","Ores and concentrates (excl. iron, manganese, ...","Ores and concentrates (excl. iron, manganese, ..."
4100,26209080,26209080,262090,"Slag, ash and residues containing metals, arse...","Ash and residues, containing metals or metal c...",Ash and residues (o/t fr the mfr of iron or st...
4110,26209970,26209970,262099,"Slag, ash and residues containing metals, arse...","Slag, ash and residues, containing metals or m...",Ash & residues containing mainly cobalt.
4425,28220000,28220000,282200,Cobalt oxides and hydroxides; commercial cobal...,Cobalt oxides and hydroxides; commercial cobal...,Cobalt oxides and hydroxides; commercial cobal...
4464,28273400,28273400,282734,"Chlorides, chloride oxides and chloride hydrox...",Cobalt chlorides,Chloride of cobalt
4470,28273930,28273930,282739,"Chlorides, chloride oxides and chloride hydrox...","Chlorides (excl. ammonium, calcium, magnesium,...",Cobalt chlorides
4472,28273985,28273985,282739,"Chlorides, chloride oxides and chloride hydrox...","Chlorides (excl. ammonium, calcium, magnesium,...","Chlorides (excl. ammonium, calcium, magnesium,..."
4509,28332930,28332930,283329,"Sulphates; alums; peroxosulphates ""persulphates""","Sulphates (excl. of sodium, magnesium, alumini...",Sulphates of cobalt and of titanium
